# Fase 2 — LoRA agente (razonamiento + MCP) — Qwen2.5-7B

**Continúa** el adapter de fase 1 (`informes`) con datasets:
- `razonamiento`
- `mcp_google_docs`

**GPU:** Runtime → T4 (o mejor).  
**Entrada:** `colab_bundle_fase2.zip` + zip del adapter fase 1.

Al final descarga `qwen25-7b-orquesta-fase2-adapter.zip` → súbelo a Modal.

## 1) GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Activa GPU: Runtime → T4"
print("CUDA OK:", torch.cuda.get_device_name(0))

## 2) Instalar Unsloth

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf
print("OK")

## 3) Subir datos fase 2

Sube `training/colab_bundle_fase2.zip`

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()  # colab_bundle_fase2.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")
ROOT = pathlib.Path("/content/training")
assert (ROOT / "data/train_messages.jsonl").exists()
print("Datos en", ROOT)

## 4) Subir adapter fase 1

Sube el zip del LoRA ya entrenado (p. ej. `qwen25-7b-informes-adapter.zip`) **o** la carpeta con `adapter_model.safetensors`.

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

ADAPTER_DIR = pathlib.Path("/content/adapter_fase1")
if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
ADAPTER_DIR.mkdir(parents=True)

uploaded = files.upload()  # adapter fase 1 .zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(ADAPTER_DIR)

# Si el zip trae una subcarpeta, ubicar adapter_config.json
cands = list(ADAPTER_DIR.rglob("adapter_config.json"))
assert cands, "No encontré adapter_config.json en el zip"
ADAPTER_PATH = cands[0].parent
print("Adapter fase 1:", ADAPTER_PATH)

## 5) Config

In [ ]:
from pathlib import Path

CFG = {
    "model": "unsloth/Qwen2.5-7B-Instruct",
    "max_seq_length": 2048,   # multi-turno MCP; bajar a 1536 si OOM
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "batch_size": 1,
    "grad_accum": 16,
    "lr": 1e-4,              # más bajo: continúa LoRA existente
    "epochs": 2,
    "seed": 42,
    "output_dir": "/content/outputs/adapter-fase2",
}
print(CFG)

## 6) Cargar base + adapter fase 1 (continuar entrenamiento)

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["model"],
    max_seq_length=CFG["max_seq_length"],
    load_in_4bit=CFG["load_in_4bit"],
    dtype=None,
)

# Continuar el LoRA de fase 1
model = PeftModel.from_pretrained(model, str(ADAPTER_PATH), is_trainable=True)
model.print_trainable_parameters()


## 7) Dataset

In [ ]:
from datasets import load_dataset

def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = load_dataset("json", data_files=str(ROOT / "data/train_messages.jsonl"), split="train").map(to_text)
val_ds = load_dataset("json", data_files=str(ROOT / "data/val_messages.jsonl"), split="train").map(to_text)
print("train", len(train_ds), "val", len(val_ds))
print(train_ds[0]["text"][:500])

## 8) Entrenar

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch, gc

gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=None,  # anti-OOM en T4
    args=SFTConfig(
        output_dir=CFG["output_dir"],
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=CFG["lr"],
        num_train_epochs=CFG["epochs"],
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=CFG["seed"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        max_seq_length=CFG["max_seq_length"],
        dataset_text_field="text",
        packing=False,
        report_to="none",
    ),
)
trainer.train()
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print("Guardado en", CFG["output_dir"])

## 9) Smoke test tool_call

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
system = (ROOT / "system_prompt_fase2.txt").read_text(encoding="utf-8")
messages = [
    {"role": "system", "content": system},
    {
        "role": "user",
        "content": (
            "Crea un documento nuevo en Google Docs titulado "
            "'Prueba Orquesta Fase 2' y prepárame la primera tool_call."
        ),
    },
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256, temperature=0.2, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## 10) Descargar adapter fase 2

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/qwen25-7b-orquesta-fase2-adapter", "zip", CFG["output_dir"])
print(zip_path)
files.download(zip_path)

## Tras Colab (local)

```bash
# Descomprime el zip en training/outputs/adapter-fase2
modal run modal/upload_adapter.py --src training/outputs/adapter-fase2
modal deploy modal/serve_informes.py
```

Si `upload_adapter.py` no tiene `--src`, copia los archivos sobre el volume como en fase 1.
